# `DocLayNet` SwiftNet-ResNet18 Architecture
**Author**: Juan Pablo Triana Martinez.

The following notebook contains all of the `torch.nn` code to recreate the
**SwiftNet** real-time segmentation architecture with a **ResNet18** backbone
from scratch (~**11.8M** parameters)!

- SwiftNet paper: "In Defense of Pre-trained ImageNet Architectures for Real-time
  Semantic Segmentation of Road-driving Images" https://arxiv.org/abs/1903.08469
- ResNet paper: https://arxiv.org/abs/1512.03385

SwiftNet is deliberately lightweight and depends on 3 components:
1. A `ResNet18Encoder` backbone.
2. A **Spatial Pyramid Pooling (SPP)** bottleneck on the 1/32 feature that
   enlarges the receptive field at negligible cost.
3. A **slim 128-channel upsampling decoder**: 2x bilinear upsample + `1x1`-projected
   lateral skip + one `3x3` blend conv per stage, back to 1/4 resolution.


## 1. The `ResNet18` encoder backbone (from scratch)

We first rebuild the **ResNet18** feature extractor from the original paper
(https://arxiv.org/abs/1512.03385), exactly as in `src/models/backbones.py`.
It is composed of:
- A **stem**: `7x7/2` convolution followed by `3x3/2` max pooling.
- Four residual stages (`layer1..layer4`), each with two `BasicBlock`s
  (two `3x3` convolutions + identity/projection skip connection).

For an input `(B, 3, 512, 512)` the encoder returns 5 multi-scale feature maps:

```python
x  -> stem_conv          -> f1 (B,  64, 256, 256)   # 1/2
f1 -> maxpool + layer1   -> f2 (B,  64, 128, 128)   # 1/4
f2 -> layer2             -> f3 (B, 128,  64,  64)   # 1/8
f3 -> layer3             -> f4 (B, 256,  32,  32)   # 1/16
f4 -> layer4             -> f5 (B, 512,  16,  16)   # 1/32
```

We start with a shared `ConvBNReLU` helper block used across all our benchmark architectures.


In [ ]:
# Let's import all necessary modules for this architecture
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class ConvBNReLU(nn.Module):
    '''
    Standard Conv2d -> BatchNorm2d -> ReLU block used across all architectures.

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        kernel_size (int): convolution kernel size.
        stride (int): convolution stride.
        padding (int): convolution padding.
        dilation (int): convolution dilation.
        groups (int): convolution groups (used for depthwise convolutions).
        relu6 (bool): if True, uses ReLU6 (MobileNetV2 convention) instead of ReLU.
    '''

    def __init__(self, m: int, n: int, kernel_size: int = 3, stride: int = 1,
                 padding: int = 1, dilation: int = 1, groups: int = 1,
                 relu6: bool = False) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=kernel_size,
                      stride=stride, padding=padding, dilation=dilation,
                      groups=groups, bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU6() if relu6 else nn.ReLU()
        )

    def forward(self, x) -> torch.Tensor:
        return self.block(x)


In [ ]:
class ResNetBasicBlock(nn.Module):
    '''
    Class that defines the BasicBlock of the ResNet18 architecture
    (two 3x3 convolutions with an identity or projected skip connection).
    Reference: https://arxiv.org/abs/1512.03385

    Args:
        m (int): number of input channels.
        n (int): number of output channels.
        stride (int): stride of the first convolution (2 halves the resolution).
    '''

    def __init__(self, m: int, n: int, stride: int = 1) -> None:
        super().__init__()

        # First 3x3 convolution (possibly downsampling)
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(3, 3),
                      stride=(stride, stride), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU()
        )

        # Second 3x3 convolution (no activation before the residual add)
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=n, out_channels=n, kernel_size=(3, 3),
                      stride=(1, 1), padding=(1, 1), bias=False),
            nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True)
        )

        # Projection skip connection when shape changes, identity otherwise
        if stride != 1 or m != n:
            self.skip_conn = nn.Sequential(
                nn.Conv2d(in_channels=m, out_channels=n, kernel_size=(1, 1),
                          stride=(stride, stride), padding=(0, 0), bias=False),
                nn.BatchNorm2d(num_features=n, eps=1e-05, momentum=0.1,
                               affine=True, track_running_stats=True)
            )
        else:
            self.skip_conn = nn.Identity()

        self.relu = nn.ReLU()

    def forward(self, x) -> torch.Tensor:
        out = self.conv_block_1(x)
        out = self.conv_block_2(out)
        out = out + self.skip_conn(x)
        return self.relu(out)


In [ ]:
class ResNet18Encoder(nn.Module):
    '''
    Class that defines the full ResNet18 feature-extractor backbone from scratch
    (no fully connected head), returning multi-scale feature maps.
    Reference: https://arxiv.org/abs/1512.03385

    Feature maps returned for an input of shape (B, Cin, H, W):
        f1: (B,  64, H/2,  W/2)   -> after stem conv (before max pooling)
        f2: (B,  64, H/4,  W/4)   -> after layer1
        f3: (B, 128, H/8,  W/8)   -> after layer2
        f4: (B, 256, H/16, W/16)  -> after layer3
        f5: (B, 512, H/32, W/32)  -> after layer4

    Args:
        Cin (int): number of input channels (3 for RGB document images).
    '''

    # Output channels at each stage, useful for building decoders
    out_channels: List[int] = [64, 64, 128, 256, 512]

    def __init__(self, Cin: int = 3) -> None:
        super().__init__()

        # Stem: 7x7/2 convolution followed by 3x3/2 max pooling
        self.stem_conv = nn.Sequential(
            nn.Conv2d(in_channels=Cin, out_channels=64, kernel_size=(7, 7),
                      stride=(2, 2), padding=(3, 3), bias=False),
            nn.BatchNorm2d(num_features=64, eps=1e-05, momentum=0.1,
                           affine=True, track_running_stats=True),
            nn.ReLU()
        )
        self.max_pool = nn.MaxPool2d(kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))

        # Four residual stages, two BasicBlocks each (ResNet18 configuration)
        self.layer1 = nn.Sequential(
            ResNetBasicBlock(m=64, n=64, stride=1),
            ResNetBasicBlock(m=64, n=64, stride=1)
        )
        self.layer2 = nn.Sequential(
            ResNetBasicBlock(m=64, n=128, stride=2),
            ResNetBasicBlock(m=128, n=128, stride=1)
        )
        self.layer3 = nn.Sequential(
            ResNetBasicBlock(m=128, n=256, stride=2),
            ResNetBasicBlock(m=256, n=256, stride=1)
        )
        self.layer4 = nn.Sequential(
            ResNetBasicBlock(m=256, n=512, stride=2),
            ResNetBasicBlock(m=512, n=512, stride=1)
        )

    def forward(self, x) -> List[torch.Tensor]:
        f1 = self.stem_conv(x)          # (B, 64, H/2, W/2)
        f2 = self.layer1(self.max_pool(f1))  # (B, 64, H/4, W/4)
        f3 = self.layer2(f2)            # (B, 128, H/8, W/8)
        f4 = self.layer3(f3)            # (B, 256, H/16, W/16)
        f5 = self.layer4(f4)            # (B, 512, H/32, W/32)
        return [f1, f2, f3, f4, f5]


### 1.1 Summary info of `ResNet18Encoder`

In [ ]:
from torchinfo import summary
# Let's inspect the ResNet18 encoder backbone
test_model = ResNet18Encoder(Cin=3)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## 2. The Spatial Pyramid Pooling (SPP) bottleneck

The 1/32 feature (512 ch) is projected to 128 channels, average-pooled onto
grids of `(8, 4, 2, 1)`, each grid projected with a `1x1` conv, upsampled back,
concatenated, and blended:

```python
f5 (B, 512, 16, 16) -> 1x1 -> (B, 128, 16, 16)
pool to 8x8 / 4x4 / 2x2 / 1x1 -> 1x1 convs -> upsample back -> concat
concat (B, 128 + 4*42, 16, 16) -> 1x1 fuse -> (B, 128, 16, 16)
```


In [ ]:
class SpatialPyramidPooling(nn.Module):
    '''
    Class that defines the SwiftNet SPP bottleneck: the 1/32 feature map is
    average-pooled onto several grid sizes, each grid is projected with a
    1x1 convolution, upsampled back, concatenated with a projection of the
    input, and blended into `n` output channels.

    Args:
        m (int): number of input channels (512 for ResNet18).
        n (int): number of output channels (128 in the paper).
        grids (tuple): pyramid grid sizes to pool onto.
        level_channels (int): channels of each pooled pyramid level.
    '''

    def __init__(self, m: int = 512, n: int = 128,
                 grids: tuple = (8, 4, 2, 1), level_channels: int = 42) -> None:
        super().__init__()
        self.grids = grids

        # 1x1 projection of the un-pooled input feature
        self.input_conv = ConvBNReLU(m=m, n=n, kernel_size=1, stride=1, padding=0)

        # One 1x1 projection per pyramid level (applied after pooling)
        self.level_convs = nn.ModuleList([
            ConvBNReLU(m=n, n=level_channels, kernel_size=1, stride=1, padding=0)
            for _ in grids
        ])

        # Final blending convolution over the concatenated pyramid
        self.fuse_conv = ConvBNReLU(m=n + len(grids) * level_channels, n=n,
                                    kernel_size=1, stride=1, padding=0)

    def forward(self, x) -> torch.Tensor:
        x = self.input_conv(x)
        levels = [x]
        for grid, conv in zip(self.grids, self.level_convs):
            pooled = F.adaptive_avg_pool2d(x, output_size=grid)
            pooled = conv(pooled)
            pooled = F.interpolate(pooled, size=x.shape[2:],
                                   mode="bilinear", align_corners=True)
            levels.append(pooled)
        return self.fuse_conv(torch.cat(levels, dim=1))


## 3. The lightweight upsampling decoder

Each module: bilinear 2x upsample, **add** a `1x1`-projected encoder skip, and
blend with a single `3x3` conv - all at a constant slim width of 128 channels:

```python
spp  (B, 128, 16, 16)  + f4 (256 -> 128) -> (B, 128,  32,  32)
                       + f3 (128 -> 128) -> (B, 128,  64,  64)
                       + f2 ( 64 -> 128) -> (B, 128, 128, 128)
1x1 head -> N logits -> 4x upsample -> (B, N, 512, 512)
```


In [ ]:
class SwiftNetUpsampleBlock(nn.Module):
    '''
    Class that defines the SwiftNet lightweight decoder module: bilinear 2x
    upsampling, addition of a 1x1-projected encoder skip connection, and one
    3x3 blending convolution.

    Args:
        skip (int): number of channels of the encoder skip feature.
        n (int): number of decoder channels (128 in the paper).
    '''

    def __init__(self, skip: int, n: int = 128) -> None:
        super().__init__()
        self.skip_conv = ConvBNReLU(m=skip, n=n, kernel_size=1, stride=1, padding=0)
        self.blend_conv = ConvBNReLU(m=n, n=n, kernel_size=3, stride=1, padding=1)

    def forward(self, x, skip) -> torch.Tensor:
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=True)
        x = x + self.skip_conv(skip)
        return self.blend_conv(x)


## 4. Final step, let's create the entire network


In [ ]:
class SwiftNetResNet18Model(nn.Module):
    '''
    Class that defines the full SwiftNet architecture with a ResNet18
    encoder, an SPP bottleneck at 1/32 resolution, and three 128-channel
    upsampling modules back to 1/4 resolution before the final 4x upsampling.

    Args:
        Cin (int): number of input channels for the encoder.
        N (int): number of output channels (1 binary / num_classes semantic).
        decoder_channels (int): width of the slim decoder (default 128).
    '''

    def __init__(self, Cin: int = 3, N: int = 1, decoder_channels: int = 128) -> None:
        super().__init__()
        self.encoder = ResNet18Encoder(Cin=Cin)
        _, c2, c3, c4, c5 = self.encoder.out_channels

        # SPP bottleneck on the 1/32 feature map
        self.spp = SpatialPyramidPooling(m=c5, n=decoder_channels)

        # Slim upsampling path with lateral skip connections
        self.upsample_16 = SwiftNetUpsampleBlock(skip=c4, n=decoder_channels)  # to 1/16
        self.upsample_8 = SwiftNetUpsampleBlock(skip=c3, n=decoder_channels)   # to 1/8
        self.upsample_4 = SwiftNetUpsampleBlock(skip=c2, n=decoder_channels)   # to 1/4

        # Segmentation head at 1/4 resolution, then 4x upsampling
        self.segmentation_head = nn.Conv2d(in_channels=decoder_channels,
                                           out_channels=N, kernel_size=(1, 1),
                                           stride=(1, 1), padding=(0, 0))

    def forward(self, x) -> torch.Tensor:
        _, f2, f3, f4, f5 = self.encoder(x)

        d = self.spp(f5)                 # (B, 128, H/32, W/32)
        d = self.upsample_16(d, f4)      # (B, 128, H/16, W/16)
        d = self.upsample_8(d, f3)       # (B, 128, H/8, W/8)
        d = self.upsample_4(d, f2)       # (B, 128, H/4, W/4)

        logits = self.segmentation_head(d)
        return F.interpolate(logits, size=x.shape[2:], mode="bilinear", align_corners=True)


### 4.1 Summary with images of shape `(B * 3 * 1024 * 1024)`

In [ ]:
from torchinfo import summary
# Full SwiftNet-ResNet18 model at 1024x1024
test_model = SwiftNetResNet18Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 1024, 1024), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


### 4.2 Summary with images of shape `(B * 3 * 512 * 512)`

In [ ]:
from torchinfo import summary
# Full SwiftNet-ResNet18 model at 512x512
test_model = SwiftNetResNet18Model(Cin=3, N=1)

summary(model = test_model,
        input_size=(1, 3, 512, 512), # (batch_size, num_channels, height, width)
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"],
        depth = 3
        )


## Integration with `src/models` and the training framework

The exact same classes above live in `src/models`, and the `swiftnet-resnet18` model can be
built through the shared model factory. This is what the training scripts use when you pass
`--arch swiftnet-resnet18`:

```bash
python scripts/train_binary_text.py --arch swiftnet-resnet18
python scripts/train_semantic_layout.py --arch swiftnet-resnet18
```

Let's double check the factory produces the same model, and run the IEEE efficiency
benchmark (parameters, FLOPs, inference latency/FPS, and peak memory - GPU if available,
otherwise CPU RSS) with `src.utils.benchmark`.


In [ ]:
import sys
from pathlib import Path
# Allow imports from the project root (src.*)
sys.path.insert(0, str(Path().cwd().parent))

from src.models import build_model

factory_model = build_model("swiftnet-resnet18", Cin=3, N=1)
total_params = sum(p.numel() for p in factory_model.parameters())
print(f"SwiftNet-ResNet18 total parameters: {total_params:,} ({total_params/1e6:.2f}M)")


In [ ]:
from src.utils import benchmark_model, print_benchmark

device = "cuda" if torch.cuda.is_available() else "cpu"
report = benchmark_model(
    model=factory_model,
    input_size=(1, 3, 512, 512),
    device=device,
    warmup=5,
    iterations=20,
    arch_name="swiftnet-resnet18",
)
print_benchmark(report)
